# 2단계: Oracle 검증기 (Colab, GPU)

정답 근거 문장(`evidence`)을 claim과 함께 넣어 SUPPORTED / NOT_SUPPORTED를 맞힌다. 검색이 완벽하다고 가정한 점수라서 **상한선**이다.
- 1단계 claim-only(test macro-F1 0.620)와의 차이 = 근거의 효과
- 나중에 5단계 end-to-end와의 차이 = 검색 때문에 잃는 성능

| 셀 | 역할 |
|---|---|
| 1 | 0단계 jsonl 불러오기. `evidence`가 없거나 빠진 문장이 있으면 여기서 멈춘다 |
| 2 | 지표 함수, 다수 클래스 기준선 |
| 3 | 토크나이즈. (claim, 근거 문장들을 이어 붙인 것)을 **문장 쌍**으로 넣는다. 길면 근거 쪽만 자른다 |
| 4 | 학습. 1단계와 같은 설정, 클래스 가중치 사용 |
| 5 | val/test 평가 |

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate

import pandas as pd
PROC = '/content/drive/MyDrive/fact-verification/data/processed'
LABELS = ['NOT_SUPPORTED', 'SUPPORTED']
dfs = {n: pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True) for n in ['train', 'val', 'test']}
for n, d in dfs.items():
    assert 'evidence' in d, '0단계 근거 추출 셀을 먼저 실행해서 evidence를 저장해야 한다'
    print(n, len(d), '근거 누락 claim', d.evidence.map(lambda ev: None in ev).sum())
    d['labels'] = d.label.map(LABELS.index)
    d['context'] = d.evidence.map(lambda ev: ' '.join(s for s in ev if s))  # 누락 문장은 빼고 이어 붙임

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train 8166 근거 누락 claim 19
val 886 근거 누락 claim 3
test 1126 근거 누락 claim 2


In [3]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def scores(y, pred):
    return {'acc': accuracy_score(y, pred), 'macro_f1': f1_score(y, pred, average='macro')}

for n in ['val', 'test']:
    y = dfs[n].labels
    print(n, 'majority(SUPPORTED)', scores(y, np.ones_like(y)))

val majority(SUPPORTED) {'acc': 0.7178329571106095, 'macro_f1': 0.4178712220762155}
test majority(SUPPORTED) {'acc': 0.4626998223801066, 'macro_f1': 0.31633272616879177}


In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
MAX_LEN = 256
tok = AutoTokenizer.from_pretrained(MODEL)

# 두 문장을 쌍으로 넣으면 토크나이저가 <s> claim </s></s> 근거 </s> 형태로 붙여 준다
ds = {n: Dataset.from_pandas(d[['claim', 'context', 'labels']].reset_index(drop=True))
            .map(lambda r: tok(r['claim'], r['context'], truncation='only_second', max_length=MAX_LEN))
      for n, d in dfs.items()}

# 잘린 비율 확인: 높으면 MAX_LEN을 늘린다
lens = [len(tok(c, x)['input_ids']) for c, x in zip(dfs['train'].claim, dfs['train'].context)]
print('토큰 길이 50/95/99%', np.percentile(lens, [50, 95, 99]), f'{MAX_LEN} 초과 {np.mean(np.array(lens) > MAX_LEN):.2%}')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Map:   0%|          | 0/8166 [00:00<?, ? examples/s]

Map:   0%|          | 0/886 [00:00<?, ? examples/s]

Map:   0%|          | 0/1126 [00:00<?, ? examples/s]

토큰 길이 50/95/99% [105. 181. 223.] 256 초과 0.31%


In [5]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

CLASS_WEIGHT = True
counts = np.bincount(dfs['train'].labels)
weight = torch.tensor(len(dfs['train']) / (2 * counts), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out = model(**inputs)
        w = weight.to(out.logits.device) if CLASS_WEIGHT else None
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=w)
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    return scores(p.label_ids, p.predictions.argmax(-1))

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir=f'/content/out/oracle_cw{int(CLASS_WEIGHT)}',
    num_train_epochs=3, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='macro_f1', report_to='none',
)
trainer = WeightedTrainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['val'],
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Acc,Macro F1
1,No log,0.414931,0.821670,0.790874
2,0.518327,0.403594,0.849887,0.820792
3,0.518327,0.384160,0.865688,0.838952


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=768, training_loss=0.4486043651898702, metrics={'train_runtime': 263.9798, 'train_samples_per_second': 92.803, 'train_steps_per_second': 2.909, 'total_flos': 2616971417707080.0, 'train_loss': 0.4486043651898702, 'epoch': 3.0})

In [6]:
from sklearn.metrics import confusion_matrix

for n in ['val', 'test']:
    p = trainer.predict(ds[n])
    pred = p.predictions.argmax(-1)
    print(n, scores(p.label_ids, pred))
    print(pd.DataFrame(confusion_matrix(p.label_ids, pred), index=[f'true_{l}' for l in LABELS], columns=[f'pred_{l}' for l in LABELS]))

val {'acc': 0.8656884875846501, 'macro_f1': 0.8389521518310612}
                    pred_NOT_SUPPORTED  pred_SUPPORTED
true_NOT_SUPPORTED                 203              47
true_SUPPORTED                      72             564


test {'acc': 0.8001776198934281, 'macro_f1': 0.8000260475248931}
                    pred_NOT_SUPPORTED  pred_SUPPORTED
true_NOT_SUPPORTED                 435             170
true_SUPPORTED                      55             466
